# Showbiz News Summarizer


This notebook builds a small **LangGraph** agent that pulls top headlines from the **News API**, turns them into structured **showbiz-style summaries** and optionally sends a digest to your phone via **Pushover**. 
A **Gradio** UI lets you pick a news category, stream pipeline logs, view summary cards, and toggle push notifications.


In [ ]:
import os
import requests
import gradio as gr
from dotenv import load_dotenv
from typing import TypedDict, List, Optional
from IPython.display import Image, display
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END, START
from newsapi import NewsApiClient

In [ ]:
load_dotenv(override=True)
serper_api_key = os.getenv("SERPER_API_KEY")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"
news_api_key = os.getenv("NEWS_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

if not serper_api_key:
    raise EnvironmentError("SERPER_API_KEY environment variable is not set")
if not pushover_user:
    raise EnvironmentError("PUSHOVER_USER environment variable is not set")
if not pushover_token:
    raise EnvironmentError("PUSHOVER_TOKEN environment variable is not set")
if not news_api_key:
    raise EnvironmentError("NEWS_API_KEY environment variable is not set")
if not openai_api_key:
    raise EnvironmentError("OPENAI_API_KEY environment variable is not set")

In [ ]:
# Define a structure output for the showbiz summary schema

class ShowbizSummary(BaseModel):
    """A structured summary of the showbiz news"""
    celebrity: str = Field(description="Main celebrity or entity.")
    headline: str = Field(description="A punchy title for the push notification.")
    summary: str = Field(description="A concise 5-sentence breakdown.")
    category: str = Field(description="Genre: e.g., Music, Film, Scandal.")
    source_link: str = Field(description="URL to the original article.")


class ShowbizNewsBatch(BaseModel):
    """Structured batch of news summaries."""
    items: List[ShowbizSummary] = Field(
        min_length=3,
        description="At least 3 summaries, one per chosen article.",
    )

In [ ]:
# Define the graph state

class AgentState(TypedDict):
    category: str
    send_push: bool
    raw_articles: List[dict]
    formatted_summaries: Optional[List[ShowbizSummary]]
    logs: str

In [ ]:
# Fetch news data with News API
def fetch_news_node(state: AgentState):
    newsapi = NewsApiClient(api_key=news_api_key)
    category = state.get("category", "entertainment")
    try:
        top_headlines = newsapi.get_top_headlines(
            category=category, language="en", page_size=20
        )
        articles = top_headlines.get("articles", [])
        return {
            "raw_articles": articles,
            "logs": f"Scanning {category} headlines ({len(articles)} articles)...",
        }
    except Exception as e:
        return {"logs": f"API Error: {str(e)}", "raw_articles": []}

In [ ]:
# News Summarizer
def summarize_node(state: AgentState):
    articles = state["raw_articles"]
    if not articles:
        return {"logs": "No news found.", "formatted_summaries": None}

    if len(articles) < 3:
        return {
            "logs": f"Need at least 3 articles; got {len(articles)}. Try another category.",
            "formatted_summaries": None,
        }

    to_summarize = articles[:15]
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    structured_output = llm.with_structured_output(ShowbizNewsBatch)
    news = []
    for i, article in enumerate(to_summarize, start=1):
        news.append(
            f"Article {i}:\nTitle: {article.get('title')}\n"
            f"Description: {article.get('description')}\n"
            f"URL: {article.get('url')}\n"
            f"Content: {article.get('content')}\n"
        )
    content = (
        "From the numbered articles below, fill `items` with at least 3 entries.\n"
        "Each item should summarize a different article from the list. "
        "Set source_link to the exact URL of the article you used.\n\n"
        + "\n---\n".join(news)
    )
    batch = structured_output.invoke([HumanMessage(content=content)])
    n = len(batch.items)
    return {
        "formatted_summaries": batch.items,
        "logs": f"Summary prepared ({n} items).",
    }

In [ ]:
# Push Notification
def push_notification_node(state: AgentState):
    if not state.get("send_push"):
        return {"logs": "Push notification skipped (User opted out)."}

    summaries = state.get("formatted_summaries") or []
    if not summaries:
        return {"logs": "Push skipped (no summaries to send)."}

    first = summaries[0]
    bullets = "\n".join(f"• {s.headline}" for s in summaries)
    payload = {
        "token": pushover_token,
        "user": pushover_user,
        "title": f"{len(summaries)} stories — {first.headline}",
        "message": f"{bullets}\n\nTop story: {first.summary}\n\nRead: {first.source_link}",
        "url": first.source_link,
    }

    response = requests.post(pushover_url, data=payload)
    status = (
        "🚀 Push notification delivered!"
        if response.status_code == 200
        else f"Push failed: {response.text}"
    )
    return {"logs": status}

In [ ]:
# Build the graph
graph_builder = StateGraph(AgentState)
graph_builder.add_node("fetcher", fetch_news_node)
graph_builder.add_node("summarizer", summarize_node)
graph_builder.add_node("notifier", push_notification_node)

graph_builder.add_edge(START, "fetcher")
graph_builder.add_edge("fetcher", "summarizer")
graph_builder.add_edge("summarizer", "notifier")
graph_builder.add_edge("notifier", END)

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Gradio
def card_html(news_summary: ShowbizSummary) -> str:
    return f"""
            <div style="border: 2px solid #6366f1; padding: 25px; border-radius: 12px; background: white; box-shadow: 0 4px 6px -1px rgb(0 0 0 / 0.1); margin-bottom: 20px;">
                <span style="background: #e0e7ff; color: #4338ca; padding: 4px 12px; border-radius: 20px; font-size: 0.8em; font-weight: bold; text-transform: uppercase;">{news_summary.category}</span>
                <h2 style="color: #111827; margin: 15px 0 10px 0;">{news_summary.headline}</h2>
                <p style="color: #4b5563; font-size: 1.1em; line-height: 1.6;">{news_summary.summary}</p>
                <hr style="border: 0; border-top: 1px solid #f3f4f6; margin: 20px 0;">
                <div style="display: flex; justify-content: space-between; align-items: center;">
                    <span style="color: #9ca3af; font-size: 0.9em;">Focus: <strong>{news_summary.celebrity}</strong></span>
                    <a href="{news_summary.source_link}" target="_blank" style="color: #6366f1; font-weight: 600; text-decoration: none;">View Source →</a>
                </div>
            </div>
            """


def run_app(category, send_push):
    state = {
        "category": category.lower(),
        "send_push": send_push,
        "raw_articles": [],
        "formatted_summaries": None,
        "logs": "Starting Showbiz News Engine...",
    }

    full_log = ""
    for output in graph.stream(state):
        for node_name, values in output.items():
            if "logs" in values:
                full_log += f"[{node_name.upper()}] {values['logs']}\n"
                yield full_log, gr.update(visible=True)

        if "summarizer" in output:
            batch = output["summarizer"].get("formatted_summaries") or []
            if batch:
                card = "".join(card_html(summary) for summary in batch)
                yield full_log, gr.update(value=card, visible=True)

with gr.Blocks(
    title="Showbiz Summarizer", theme=gr.themes.Default(primary_hue="indigo")
) as demo:
    gr.Markdown("# 🎬 Showbiz News Summarizer")

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group():
                cat_input = gr.Dropdown(
                    choices=[
                        "Entertainment",
                        "General",
                        "Technology",
                        "Science",
                        "Sports",
                    ],
                    value="Entertainment",
                    label="Select Category",
                )
                push_toggle = gr.Checkbox(
                    label="Send Push Notification to Phone", value=False
                )
                run_btn = gr.Button("Fetch News", variant="primary")

            log_box = gr.Textbox(label="Agent Status Logs", lines=10, interactive=False)

        with gr.Column(scale=2):
            display_area = gr.HTML(visible=False)

    run_btn.click(
        fn=run_app, inputs=[cat_input, push_toggle], outputs=[log_box, display_area]
    )
    demo.launch(inbrowser=True)